In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main')
!ls eval/

Mounted at /content/drive
dataset.py	eval_cityscapes_color.py   iouEval.py	transform.py
erfnet_nobn.py	eval_cityscapes_server.py  __pycache__
erfnet.py	eval_forwardTime.py	   README.md
evalAnomaly.py	eval_iou.py		   results.txt


In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/Anomaly_Validation_Datasets.zip"
extract_path = "/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/"

with zipfile.ZipFile(zip_path) as z:
    z.extractall(extract_path)

print("Estratto!")

Estratto!


In [ ]:
!pip install ood-metrics scikit-learn -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 125.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompa

In [ ]:
import os
os.chdir('/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/eval')

!python evalAnomaly.py \
    --input "/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/Validation_Dataset/RoadAnomaly21/images/*.png" \
    --loadDir "/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/trained_models/" \
    --loadWeights "erfnet_pretrained.pth"

Loading model: /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/trained_models/erfnet.py
Loading weights: /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/trained_models/erfnet_pretrained.pth
Model and weights LOADED successfully
/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/Validation_Dataset/RoadAnomaly21/images/8.png
Traceback (most recent call last):
  File "/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/eval/evalAnomaly.py", line 163, in <module>
    main()
  File "/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/eval/evalAnomaly.py", line 102, in main
    result = model(images)
             ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", 

In [ ]:
with open('evalAnomaly.py', 'r') as f:
    content = f.read()

content = content.replace(
    'images = images.permute(0,3,1,2)',
    '# images = images.permute(0,3,1,2)  # removed: ToTensor already gives [C,H,W]'
)

with open('evalAnomaly_fixed.py', 'w') as f:
    f.write(content)

print("Fatto!")

Fatto!


In [ ]:
!python evalAnomaly_fixed.py \
    --input "/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/Validation_Dataset/RoadAnomaly21/images/*.png" \
    --loadDir "/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/trained_models/" \
    --loadWeights "erfnet_pretrained.pth"

Loading model: /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/trained_models/erfnet.py
Loading weights: /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/trained_models/erfnet_pretrained.pth
Model and weights LOADED successfully
/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/Validation_Dataset/RoadAnomaly21/images/8.png
/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/Validation_Dataset/RoadAnomaly21/images/9.png
/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/Validation_Dataset/RoadAnomaly21/images/4.png
/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/Validation_Dataset/RoadAnomaly21/images/5.png
/content/drive/MyDrive/MaskArc

In [ ]:
%%writefile evalAnomaly_all.py
# Copyright (c) OpenMMLab. All rights reserved.
# Modified to support MSP, MaxLogit, and MaxEntropy methods
import os
import cv2
import glob
import torch
import random
from PIL import Image
import numpy as np
from erfnet import ERFNet
import os.path as osp
from argparse import ArgumentParser
from ood_metrics import fpr_at_95_tpr, calc_metrics, plot_roc, plot_pr, plot_barcode
from sklearn.metrics import roc_auc_score, roc_curve, auc, precision_recall_curve, average_precision_score
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
# AGGIUNTO: importiamo functional per softmax e log_softmax
import torch.nn.functional as F

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

NUM_CHANNELS = 3
NUM_CLASSES = 20
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

input_transform = Compose(
    [
        Resize((512, 1024), Image.BILINEAR),
        ToTensor(),
    ]
)

target_transform = Compose(
    [
        Resize((512, 1024), Image.NEAREST),
    ]
)


def main():
    parser = ArgumentParser()
    parser.add_argument(
        "--input",
        default="/home/shyam/Mask2Former/unk-eval/RoadObsticle21/images/*.webp",
        nargs="+",
        help="A list of space separated input images; "
        "or a single glob pattern such as 'directory/*.jpg'",
    )
    parser.add_argument('--loadDir', default="../trained_models/")
    parser.add_argument('--loadWeights', default="erfnet_pretrained.pth")
    parser.add_argument('--loadModel', default="erfnet.py")
    parser.add_argument('--subset', default="val")
    parser.add_argument('--datadir', default="/home/shyam/ViT-Adapter/segmentation/data/cityscapes/")
    parser.add_argument('--num-workers', type=int, default=4)
    parser.add_argument('--batch-size', type=int, default=1)
    parser.add_argument('--cpu', action='store_true')
    # AGGIUNTO: argomento per scegliere il metodo di anomaly detection
    parser.add_argument('--method', default="maxlogit", choices=["msp", "maxlogit", "maxentropy"],
                        help="Anomaly scoring method: msp, maxlogit, or maxentropy")
    args = parser.parse_args()

    anomaly_score_list = []
    ood_gts_list = []

    if not os.path.exists('results.txt'):
        open('results.txt', 'w').close()
    file = open('results.txt', 'a')

    modelpath = args.loadDir + args.loadModel
    weightspath = args.loadDir + args.loadWeights

    print("Loading model: " + modelpath)
    print("Loading weights: " + weightspath)
    # AGGIUNTO: stampa il metodo scelto
    print("Anomaly method: " + args.method)

    model = ERFNet(NUM_CLASSES)

    if not args.cpu:
        model = torch.nn.DataParallel(model).cuda()

    def load_my_state_dict(model, state_dict):
        own_state = model.state_dict()
        for name, param in state_dict.items():
            if name not in own_state:
                if name.startswith("module."):
                    own_state[name.split("module.")[-1]].copy_(param)
                else:
                    print(name, " not loaded")
                    continue
            else:
                own_state[name].copy_(param)
        return model

    model = load_my_state_dict(model, torch.load(weightspath, map_location=lambda storage, loc: storage))
    print("Model and weights LOADED successfully")
    model.eval()

    for path in glob.glob(os.path.expanduser(str(args.input[0]))):
        print(path)
        images = input_transform((Image.open(path).convert('RGB'))).unsqueeze(0).float().cuda()
        # RIMOSSO: images = images.permute(0,3,1,2) — ToTensor() restituisce già [C,H,W]

        with torch.no_grad():
            result = model(images)  # output shape: [1, NUM_CLASSES, H, W]

        # AGGIUNTO: calcolo dell'anomaly score in base al metodo scelto
        if args.method == "maxlogit":
            # MaxLogit: usa direttamente i logit grezzi — score alto = anomalia
            # Già implementato nell'originale
            anomaly_result = 1.0 - np.max(result.squeeze(0).data.cpu().numpy(), axis=0)

        elif args.method == "msp":
            # MSP (Maximum Softmax Probability): applica softmax ai logit
            # Score = 1 - max(softmax) — bassa confidenza = possibile anomalia
            probs = F.softmax(result.squeeze(0), dim=0)
            anomaly_result = 1.0 - np.max(probs.data.cpu().numpy(), axis=0)

        elif args.method == "maxentropy":
            # MaxEntropy: usa l'entropia della distribuzione softmax
            # Alta entropia = il modello è incerto = possibile anomalia
            probs = F.softmax(result.squeeze(0), dim=0)
            # Entropia = -sum(p * log(p)) per ogni pixel
            log_probs = torch.log(probs + 1e-9)  # +1e-9 per evitare log(0)
            entropy = -torch.sum(probs * log_probs, dim=0)
            anomaly_result = entropy.data.cpu().numpy()

        pathGT = path.replace("images", "labels_masks")
        if "RoadObsticle21" in pathGT:
            pathGT = pathGT.replace("webp", "png")
        if "fs_static" in pathGT:
            pathGT = pathGT.replace("jpg", "png")
        if "RoadAnomaly" in pathGT:
            pathGT = pathGT.replace("jpg", "png")

        mask = Image.open(pathGT)
        mask = target_transform(mask)
        ood_gts = np.array(mask)

        if "RoadAnomaly" in pathGT:
            ood_gts = np.where((ood_gts == 2), 1, ood_gts)
        if "LostAndFound" in pathGT:
            ood_gts = np.where((ood_gts == 0), 255, ood_gts)
            ood_gts = np.where((ood_gts == 1), 0, ood_gts)
            ood_gts = np.where((ood_gts > 1) & (ood_gts < 201), 1, ood_gts)
        if "Streethazard" in pathGT:
            ood_gts = np.where((ood_gts == 14), 255, ood_gts)
            ood_gts = np.where((ood_gts < 20), 0, ood_gts)
            ood_gts = np.where((ood_gts == 255), 1, ood_gts)

        if 1 not in np.unique(ood_gts):
            continue
        else:
            ood_gts_list.append(ood_gts)
            anomaly_score_list.append(anomaly_result)
        del result, anomaly_result, ood_gts, mask
        torch.cuda.empty_cache()

    file.write("\n")

    ood_gts = np.array(ood_gts_list)
    anomaly_scores = np.array(anomaly_score_list)

    ood_mask = (ood_gts == 1)
    ind_mask = (ood_gts == 0)

    ood_out = anomaly_scores[ood_mask]
    ind_out = anomaly_scores[ind_mask]

    ood_label = np.ones(len(ood_out))
    ind_label = np.zeros(len(ind_out))

    val_out = np.concatenate((ind_out, ood_out))
    val_label = np.concatenate((ind_label, ood_label))

    prc_auc = average_precision_score(val_label, val_out)
    fpr = fpr_at_95_tpr(val_out, val_label)

    print(f'AUPRC score: {prc_auc * 100.0}')
    print(f'FPR@TPR95: {fpr * 100.0}')

    # AGGIUNTO: scrive anche il metodo usato nel file risultati
    file.write(f'Method: {args.method}    AUPRC score: {prc_auc * 100.0}   FPR@TPR95: {fpr * 100.0}')
    file.close()


if __name__ == '__main__':
    main()

Writing evalAnomaly_all.py


In [ ]:
!python evalAnomaly_all.py \
    --input "/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/Validation_Dataset/RoadAnomaly21/images/*.png" \
    --loadDir "/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/trained_models/" \
    --loadWeights "erfnet_pretrained.pth" \
    --method maxlogit

Loading model: /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/trained_models/erfnet.py
Loading weights: /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/trained_models/erfnet_pretrained.pth
Anomaly method: maxlogit
Model and weights LOADED successfully
/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/Validation_Dataset/RoadAnomaly21/images/8.png
/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/Validation_Dataset/RoadAnomaly21/images/9.png
/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/Validation_Dataset/RoadAnomaly21/images/4.png
/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/Validation_Dataset/RoadAnomaly21/images/5.png
/cont

In [ ]:
%%writefile run_all_erfnet.sh
#!/bin/bash

DATASETS_PATH="/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/Validation_Dataset"
MODELS_PATH="/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/trained_models"

for METHOD in msp maxlogit maxentropy; do
    echo "===== METHOD: $METHOD ====="

    echo "--- RoadAnomaly21 ---"
    python evalAnomaly_all.py --input "$DATASETS_PATH/RoadAnomaly21/images/*.png" \
        --loadDir "$MODELS_PATH/" --loadWeights "erfnet_pretrained.pth" --method $METHOD

    echo "--- RoadObsticle21 ---"
    python evalAnomaly_all.py --input "$DATASETS_PATH/RoadObsticle21/images/*.webp" \
        --loadDir "$MODELS_PATH/" --loadWeights "erfnet_pretrained.pth" --method $METHOD

    echo "--- fs_static ---"
    python evalAnomaly_all.py --input "$DATASETS_PATH/fs_static/images/*.jpg" \
        --loadDir "$MODELS_PATH/" --loadWeights "erfnet_pretrained.pth" --method $METHOD

    echo "--- FS_LostFound_full ---"
    python evalAnomaly_all.py --input "$DATASETS_PATH/FS_LostFound_full/images/*.png" \
        --loadDir "$MODELS_PATH/" --loadWeights "erfnet_pretrained.pth" --method $METHOD

    echo "--- RoadAnomaly ---"
    python evalAnomaly_all.py --input "$DATASETS_PATH/RoadAnomaly/images/*.jpg" \
        --loadDir "$MODELS_PATH/" --loadWeights "erfnet_pretrained.pth" --method $METHOD
done

Writing run_all_erfnet.sh


In [ ]:
!bash run_all_erfnet.sh

===== METHOD: msp =====
--- RoadAnomaly21 ---
Loading model: /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/trained_models/erfnet.py
Loading weights: /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/trained_models/erfnet_pretrained.pth
Anomaly method: msp
Model and weights LOADED successfully
/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/Validation_Dataset/RoadAnomaly21/images/8.png
/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/Validation_Dataset/RoadAnomaly21/images/9.png
/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/Validation_Dataset/RoadAnomaly21/images/4.png
/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/Validation

In [ ]:
import zipfile

base = "/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/"

# Estrai solo il val set delle immagini (evita di estrarre tutto il training set)
print("Estraendo leftImg8bit val...")
with zipfile.ZipFile(base + "leftImg8bit_trainvaltest.zip") as z:
    val_files = [f for f in z.namelist() if "leftImg8bit/val" in f]
    z.extractall(base, members=val_files)

print("Estraendo gtFine val...")
with zipfile.ZipFile(base + "gtFine_trainvaltest.zip") as z:
    val_files = [f for f in z.namelist() if "gtFine/val" in f]
    z.extractall(base, members=val_files)

print("Fatto!")

Estraendo leftImg8bit val...
Estraendo gtFine val...
Fatto!


In [ ]:
import os
gtfine_path = "/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/gtFine/val/frankfurt"
print(os.listdir(gtfine_path)[:5])

['frankfurt_000001_021406_gtFine_labelIds.png', 'frankfurt_000001_059789_gtFine_labelIds.png', 'frankfurt_000001_020046_gtFine_instanceIds.png', 'frankfurt_000001_032711_gtFine_labelIds.png', 'frankfurt_000001_025713_gtFine_polygons.json']


In [ ]:
!pip install cityscapesscripts -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 473.6/473.6 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 11.9 MB/s eta 0:00:00


In [ ]:
import os
os.environ["CITYSCAPES_DATASET"] = "/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main"

!python -m cityscapesscripts.preparation.createTrainIdLabelImgs

Processing 500 annotation files
Progress: 100.0 % 

In [ ]:
!python eval_iou.py \
    --datadir /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main \
    --subset val

Loading model: ../trained_models/erfnet.py
Loading weights: ../trained_models/erfnet_pretrained.pth
Model and weights LOADED successfully
/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/leftImg8bit/val /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject-main/MaskArchitectureAnomaly_CourseProject-main/gtFine/val
0 val/frankfurt/frankfurt_000000_000294_leftImg8bit.png
1 val/frankfurt/frankfurt_000000_000576_leftImg8bit.png
2 val/frankfurt/frankfurt_000000_001016_leftImg8bit.png
3 val/frankfurt/frankfurt_000000_001236_leftImg8bit.png
4 val/frankfurt/frankfurt_000000_001751_leftImg8bit.png
5 val/frankfurt/frankfurt_000000_002196_leftImg8bit.png
6 val/frankfurt/frankfurt_000000_002963_leftImg8bit.png
7 val/frankfurt/frankfurt_000000_003025_leftImg8bit.png
8 val/frankfurt/frankfurt_000000_003357_leftImg8bit.png
9 val/frankfurt/frankfurt_000000_003920_leftImg8bit.png
10 val/frankfurt/frankfurt_000000_004617_leftImg8bit.png
